# 02 — Dataset Preparation & 5-Fold Split

**Skripsi:** Klasifikasi Tumor Paru pada Citra CT Scan Menggunakan DenseNet121
dan Explainable Artificial Intelligence berbasis Grad-CAM.

Notebook ini menyiapkan **metadata dataset** dan **pembagian 5-fold yang bebas
data leakage** untuk tahap training DenseNet121 + Grad-CAM.

| Aspek | Nilai |
|---|---|
| Dataset | LIDC-IDRI |
| Klasifikasi | benign vs malignant (binary) |
| Validasi | 5-Fold Cross Validation |
| Grouping | `PatientID` (satu pasien tidak boleh terbagi antar fold) |
| Reproducibility | `random_state = 42` |

### Batas kerja notebook ini

Notebook ini **hanya** bekerja pada tabel metadata. Yang **tidak** ada di sini:

- preprocessing DICOM (ada di `01_preprocessing.ipynb`)
- crop / resize / augmentation
- batch generator, training DenseNet121
- perhitungan metrik model
- oversampling / undersampling
- Grad-CAM

### Sumber data

`dataset/02_annotations/pylidc_nodules_extracted.xlsx` — hasil ekstraksi
anotasi. Notebook mencari file ini secara portable lewat `PROJECT_ROOT`.

## 1. Configuration

Semua path diturunkan dari `PROJECT_ROOT` agar notebook portabel antara
laptop, PC Lab, dan `git clone` baru.

In [1]:
# --- CELL 1.1 - IMPORTS -------------------------------------------------
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from IPython.display import display
except ImportError:                      # di luar Jupyter
    def display(obj):
        print(obj)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


In [2]:
# --- CELL 1.2 - KONFIGURASI ----------------------------------------------
# --- PARAMETER SPLIT (nilai reproduibel, jangan diubah tanpa alasan) ------
N_SPLITS = 5
RANDOM_STATE = 42
SHUFFLE = True

# --- ATURAN LABEL (dikunci, sama dengan project) -------------------------
# Mean Consensus Malignancy:
#   <= 2.5      -> Benign    (0)
#   2.5 .. 3.5  -> Exclude   (tidak dipakai untuk training)
#   >= 3.5      -> Malignant (1)
LABEL_THRESHOLD_BENIGN_MAX = 2.5
LABEL_THRESHOLD_MALIGNANT_MIN = 3.5
LABEL_BENIGN = "Benign"
LABEL_MALIGNANT = "Malignant"
LABEL_EXCLUDE = "Exclude"
LABEL_BINARY_MAP = {LABEL_BENIGN: 0, LABEL_MALIGNANT: 1}

# Ekspektasi hasil ekstraksi sebelumnya. TIDAK dipaksakan: kalau berbeda,
# notebook menampilkan angka aktual dan memberi peringatan.
EXPECTED_BENIGN = 381
EXPECTED_MALIGNANT = 143
EXPECTED_CANDIDATES = 524

# --- PATH (relatif terhadap PROJECT_ROOT) --------------------------------
ANNOTATION_RELPATH = Path("dataset/02_annotations/pylidc_nodules_extracted.xlsx")
SPLIT_RELDIR = Path("dataset/05_splits")

MANIFEST_FILENAME = "nodule_manifest.csv"
FOLD_ASSIGN_FILENAME = "fold_assignments.csv"
FOLD_SUMMARY_FILENAME = "fold_summary.csv"
SPLIT_CONFIG_FILENAME = "split_config.json"


def find_project_root(marker: Path, start: Path | None = None) -> Path:
    """Cari folder repository yang memuat `marker`, tanpa path hard-code."""
    base = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [base, *base.parents]:
        if (candidate / marker).exists():
            return candidate
    searched = "\n".join(f"  - {p}" for p in [base, *base.parents])
    raise FileNotFoundError(
        f"PROJECT_ROOT tidak ditemukan.\n"
        f"Notebook mencari folder yang memuat: {marker}\n\n"
        f"Direktori yang sudah diperiksa:\n{searched}\n\n"
        f"Pastikan notebook dijalankan dari dalam repository."
    )


PROJECT_ROOT = find_project_root(ANNOTATION_RELPATH)
ANNOTATION_FILE = PROJECT_ROOT / ANNOTATION_RELPATH
SPLIT_DIR = PROJECT_ROOT / SPLIT_RELDIR

print("KONFIGURASI")
print("-" * 70)
print(f"Python         : {sys.version.split()[0]}")
print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"Input          : {ANNOTATION_RELPATH}")
print(f"Output dir     : {SPLIT_RELDIR}")
print(f"N_SPLITS       : {N_SPLITS}")
print(f"RANDOM_STATE   : {RANDOM_STATE}")
print(f"SHUFFLE        : {SHUFFLE}")
print(f"Aturan label   : <= {LABEL_THRESHOLD_BENIGN_MAX} Benign | "
      f"> {LABEL_THRESHOLD_BENIGN_MAX} dan < {LABEL_THRESHOLD_MALIGNANT_MIN} Exclude | "
      f">= {LABEL_THRESHOLD_MALIGNANT_MIN} Malignant")
print(f"Grouping       : PatientID")


KONFIGURASI
----------------------------------------------------------------------
Python         : 3.12.10
PROJECT_ROOT   : C:\230016_rivan\source-code2020
Input          : dataset\02_annotations\pylidc_nodules_extracted.xlsx
Output dir     : dataset\05_splits
N_SPLITS       : 5
RANDOM_STATE   : 42
SHUFFLE        : True
Aturan label   : <= 2.5 Benign | > 2.5 dan < 3.5 Exclude | >= 3.5 Malignant
Grouping       : PatientID


## 2. Load Annotation Metadata

Membaca tabel anotasi hasil ekstraksi `pylidc`. Notebook ini **tidak**
mengimpor ulang dari `pylidc` dan **tidak** mengubah file sumber.

In [3]:
# --- CELL 2 - LOAD ANNOTATION --------------------------------------------
if not ANNOTATION_FILE.is_file():
    raise FileNotFoundError(
        f"File annotation tidak ditemukan:\n  {ANNOTATION_FILE}\n\n"
        f"File ini dihasilkan oleh scripts/03_extract_pylidc_nodules.py\n"
        f"dan WAJIB ada di repository."
    )

ann = pd.read_excel(ANNOTATION_FILE)

print(f"File  : {ANNOTATION_RELPATH}")
print(f"Baris : {len(ann)}")
print(f"Kolom : {len(ann.columns)}")
print("\nNama kolom:")
for i, col in enumerate(ann.columns):
    print(f"  [{i:2}] {col}  ({ann[col].dtype})")

print("\n5 baris pertama:")
display(ann.head(5))


File  : dataset\02_annotations\pylidc_nodules_extracted.xlsx
Baris : 837
Kolom : 18

Nama kolom:
  [ 0] PatientID  (object)
  [ 1] StudyInstanceUID  (object)
  [ 2] SeriesInstanceUID  (object)
  [ 3] NoduleID  (object)
  [ 4] ClusterIndex  (int64)
  [ 5] NReaders  (int64)
  [ 6] MalignancyScores  (object)
  [ 7] MeanMalignancy  (float64)
  [ 8] Label  (object)
  [ 9] LabelCode  (float64)
  [10] AnnotationIDs  (object)
  [11] BBox_X_Min  (int64)
  [12] BBox_X_Max  (int64)
  [13] BBox_Y_Min  (int64)
  [14] BBox_Y_Max  (int64)
  [15] BBox_Z_Min  (int64)
  [16] BBox_Z_Max  (int64)
  [17] BBox_Z_Slices  (int64)

5 baris pertama:


,PatientID,StudyInstanceUID,SeriesInstanceUID,NoduleID,ClusterIndex,NReaders,MalignancyScores,MeanMalignancy,Label,LabelCode,AnnotationIDs,BBox_X_Min,BBox_X_Max,BBox_Y_Min,BBox_Y_Max,BBox_Z_Min,BBox_Z_Max,BBox_Z_Slices
0,LIDC-IDRI-0001,1.3.6.1.4.1.14519.5.2.1.6279.6001.298806137288...,1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636...,1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636...,1,4,"5,5,5,4",4.75,Malignant,1.0,"84,85,86,87",340,391,297,340,86,94,9
1,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,1,1,2,2.00,Benign,0.0,102,331,366,350,383,62,69,8
2,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,2,4,"5,5,3,4",4.25,Malignant,1.0,"90,93,98,99",333,372,344,388,70,79,10
3,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,3,4,"4,4,3,2",3.25,Exclude,NaN,"91,95,97,100",192,204,299,317,81,84,4
4,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,1.3.6.1.4.1.14519.5.2.1.6279.6001.170706757615...,4,4,"4,5,3,2",3.50,Malignant,1.0,"92,94,96,101",209,228,212,230,81,88,8


In [4]:
# --- CELL 2.1 - KEKOSONGAN, LABEL, MALIGNANCY, NREADERS ------------------
null_counts = ann.isna().sum()
null_table = pd.DataFrame({
    "kolom": null_counts.index,
    "jumlah_kosong": null_counts.to_numpy(),
    "persen": (null_counts / len(ann) * 100).round(2).to_numpy(),
})
print("JUMLAH NILAI KOSONG PER KOLOM")
print("-" * 70)
display(null_table)

print("\nDistribusi Label:")
display(ann["Label"].value_counts().rename_axis("Label").to_frame("jumlah"))

print("\nDistribusi MeanMalignancy:")
display(ann["MeanMalignancy"].describe().round(3).to_frame("nilai"))

print("\nDistribusi NReaders:")
display(ann["NReaders"].value_counts().sort_index().rename_axis("NReaders").to_frame("jumlah"))


JUMLAH NILAI KOSONG PER KOLOM
----------------------------------------------------------------------


,kolom,jumlah_kosong,persen
0,PatientID,0,0.0
1,StudyInstanceUID,0,0.0
2,SeriesInstanceUID,0,0.0
3,NoduleID,0,0.0
4,ClusterIndex,0,0.0
5,NReaders,0,0.0
6,MalignancyScores,0,0.0
7,MeanMalignancy,0,0.0
8,Label,0,0.0
9,LabelCode,313,37.4



Distribusi Label:


,jumlah
Label,
Benign,381
Exclude,313
Malignant,143



Distribusi MeanMalignancy:


,nilai
count,837.000
mean,2.661
std,0.894
min,1.000
25%,2.000
50%,2.667
75%,3.000
max,5.000



Distribusi NReaders:


,jumlah
NReaders,
1,262
2,159
3,152
4,260
5,2
7,1
8,1


## 3. Metadata Validation

Memastikan kolom identitas tersedia dan konsisten **sebelum** membuat split.

Grouping untuk cross-validation bergantung pada `PatientID`. Karena itu
`PatientID` bersifat kritis: jika kosong, salah, atau tidak konsisten dengan
`SeriesInstanceUID`, seluruh pembagian fold menjadi tidak sah.

In [5]:
# --- CELL 3 - VALIDASI KOLOM WAJIB ---------------------------------------
# Kolom yang dibutuhkan. NoduleClusterID tidak ada di file sumber, sehingga
# notebook membuat kolom itu dari NoduleID pada cell berikutnya.
REQUIRED_COLUMNS = [
    "PatientID",
    "SeriesInstanceUID",
    "NoduleClusterID",
    "MeanMalignancy",
    "NReaders",
    "Label",
    "BBox_X_Min", "BBox_X_Max",
    "BBox_Y_Min", "BBox_Y_Max",
    "BBox_Z_Min", "BBox_Z_Max",
    "BBox_Z_Slices",
]

# Sumber untuk NoduleClusterID. File pylidc_nodules_extracted.xlsx tidak
# memuat kolom NoduleClusterID; kolom yang merepresentasikan satu baris
# annotation (satu nodul) adalah NoduleID, yang nilainya unik 837/837.
CLUSTER_ID_SOURCE = "NoduleID"
REQUIRED_COLUMNS_FROM_SOURCE = [c for c in REQUIRED_COLUMNS
                                if c != "NoduleClusterID"]

missing = [c for c in REQUIRED_COLUMNS_FROM_SOURCE
           if c not in ann.columns]
if missing:
    raise KeyError(
        f"Kolom wajib tidak ditemukan di annotation:\n  {missing}\n"
        f"Kolom yang tersedia: {list(ann.columns)}"
    )
print("Semua kolom wajib tersedia (selain NoduleClusterID, lihat di bawah).")

if "PatientID" not in ann.columns:
    raise KeyError(
        "PatientID tidak tersedia.\n\n"
        "PatientID adalah GROUPING VARIABLE untuk 5-fold cross validation.\n"
        "Menggantinya dengan SeriesInstanceUID TIDAK dilakukan otomatis karena\n"
        "perannya berbeda: satu pasien dapat memiliki beberapa seri.\n\n"
        "Perbaiki sumber data, atau setujui penggantian secara eksplisit."
    )

# NoduleClusterID dibuat dari NoduleID (keputusan eksplisit, bukan alias diam-diam)
if CLUSTER_ID_SOURCE not in ann.columns:
    raise KeyError(f"Kolom sumber '{CLUSTER_ID_SOURCE}' tidak ada untuk "
                   f"membentuk NoduleClusterID.")
ann["NoduleClusterID"] = ann[CLUSTER_ID_SOURCE].astype(str).str.strip()

print(f"NoduleClusterID  <- {CLUSTER_ID_SOURCE}")
print(f"  nilai unik     : {ann['NoduleClusterID'].nunique()} dari {len(ann)} baris")
print(f"  Catatan        : file sumber tidak punya kolom NoduleClusterID.")
print(f"                   Kolom itu dibentuk dari {CLUSTER_ID_SOURCE} yang")
print(f"                   merepresentasikan satu baris = satu nodul.")


Semua kolom wajib tersedia (selain NoduleClusterID, lihat di bawah).
NoduleClusterID  <- NoduleID
  nilai unik     : 837 dari 837 baris
  Catatan        : file sumber tidak punya kolom NoduleClusterID.
                   Kolom itu dibentuk dari NoduleID yang
                   merepresentasikan satu baris = satu nodul.


In [6]:
# --- CELL 3.1 - KONSISTENSI IDENTITAS ------------------------------------
identity_problems = []

# (a) kolom identitas tidak boleh kosong
for col in ("PatientID", "SeriesInstanceUID", "NoduleClusterID"):
    n_null = int(ann[col].isna().sum())
    n_blank = int((ann[col].astype(str).str.strip() == "").sum())
    if n_null or n_blank:
        identity_problems.append(
            f"{col}: {n_null} null + {n_blank} string kosong")
    else:
        print(f"[OK ] {col:<20} tidak kosong "
              f"({ann[col].nunique()} nilai unik)")

# (b) satu NoduleClusterID muncul lebih dari sekali?
dup_cluster = int(ann["NoduleClusterID"].duplicated().sum())
print(f"\nNoduleClusterID muncul >1 kali : {dup_cluster}")
if dup_cluster:
    identity_problems.append(
        f"NoduleClusterID duplikat pada {dup_cluster} baris")

# (c) satu SeriesInstanceUID -> satu PatientID secara konsisten?
seri_multi_patient = (ann.groupby("SeriesInstanceUID")["PatientID"]
                      .nunique())
n_seri_bad = int((seri_multi_patient > 1).sum())
print(f"Seri dengan >1 PatientID       : {n_seri_bad}")
if n_seri_bad:
    identity_problems.append(
        f"{n_seri_bad} SeriesInstanceUID terkait lebih dari satu PatientID")

# (d) satu PatientID boleh memiliki beberapa seri (WAJAR, bukan error)
patient_multi_series = (ann.groupby("PatientID")["SeriesInstanceUID"]
                        .nunique())
n_patient_multi = int((patient_multi_series > 1).sum())
print(f"PatientID dengan >1 seri       : {n_patient_multi}  "
      f"(wajar, bukan error)")
if n_patient_multi:
    print(f"  Contoh: {patient_multi_series[patient_multi_series > 1].head(3).to_dict()}")
print(f"\nHubungan PatientID <-> SeriesInstanceUID : "
      f"{'1:1' if n_patient_multi == 0 else '1:N'}")

print("\nRingkasan identitas:")
display(pd.DataFrame({
    "entitas": ["baris annotation", "PatientID", "SeriesInstanceUID",
                "NoduleClusterID"],
    "jumlah_unik": [len(ann), ann["PatientID"].nunique(),
                    ann["SeriesInstanceUID"].nunique(),
                    ann["NoduleClusterID"].nunique()],
}))

if identity_problems:
    print("\n" + "=" * 70)
    print("IDENTITAS DATA TIDAK KONSISTEN - SPLIT DIBATALKAN")
    print("=" * 70)
    for p in identity_problems:
        print(f"  - {p}")
    raise ValueError(
        "Identitas data bermasalah. Perbaiki sumber data sebelum membuat split. "
        "Detail di atas."
    )
print("\nIdentitas data konsisten. Lanjut ke filter label.")


[OK ] PatientID            tidak kosong (269 nilai unik)
[OK ] SeriesInstanceUID    tidak kosong (269 nilai unik)
[OK ] NoduleClusterID      tidak kosong (837 nilai unik)

NoduleClusterID muncul >1 kali : 0
Seri dengan >1 PatientID       : 0
PatientID dengan >1 seri       : 0  (wajar, bukan error)

Hubungan PatientID <-> SeriesInstanceUID : 1:1

Ringkasan identitas:


,entitas,jumlah_unik
0,baris annotation,837
1,PatientID,269
2,SeriesInstanceUID,269
3,NoduleClusterID,837



Identitas data konsisten. Lanjut ke filter label.


## 4. Label Filtering

Aturan label (**tidak diubah**) berdasarkan Mean Consensus Malignancy:

```
MeanMalignancy <= 2.5      -> Benign    (LabelBinary = 0)
2.5 < MeanMalignancy < 3.5 -> Exclude   (dibuang)
MeanMalignancy >= 3.5      -> Malignant (LabelBinary = 1)
```

Hanya `Exclude` yang dibuang. `Benign` dan `Malignant` dipertahankan.

In [7]:
# --- CELL 4 - FILTER LABEL -----------------------------------------------
def label_from_malignancy(mean_malignancy: float) -> str:
    """Aturan label project. JANGAN diubah tanpa persetujuan."""
    if pd.isna(mean_malignancy):
        return "NO_SCORE"
    if mean_malignancy <= LABEL_THRESHOLD_BENIGN_MAX:
        return LABEL_BENIGN
    if mean_malignancy >= LABEL_THRESHOLD_MALIGNANT_MIN:
        return LABEL_MALIGNANT
    return LABEL_EXCLUDE


# konsistensi kolom Label yang ada terhadap aturan
derived = ann["MeanMalignancy"].map(label_from_malignancy)
mismatch = int((derived != ann["Label"]).sum())
print(f"Baris tidak cocok dengan aturan label : {mismatch} dari {len(ann)}")
if mismatch:
    display(ann.loc[derived != ann["Label"],
                    ["PatientID", "NoduleClusterID", "MeanMalignancy", "Label"]]
            .assign(Label_dari_aturan=derived[derived != ann["Label"]]))
    raise ValueError(
        f"Kolom Label tidak konsisten dengan aturan threshold pada {mismatch} "
        f"baris. Periksa sumber data sebelum filter."
    )
print("Kolom Label konsisten dengan aturan threshold.")

# kandidat = semua baris, label Exclude ditandai (tidak dihapus di sini)
df = ann.copy()
df["LabelBinary"] = df["Label"].map(LABEL_BINARY_MAP)   # Exclude -> NaN

counts = df["Label"].value_counts()
n_benign = int(counts.get(LABEL_BENIGN, 0))
n_malignant = int(counts.get(LABEL_MALIGNANT, 0))
n_exclude = int(counts.get(LABEL_EXCLUDE, 0))
n_candidates = n_benign + n_malignant

print("\nFilter label")
print("-" * 70)
print(f"Total kandidat (semua baris) : {len(df)}")
print(f"  {LABEL_BENIGN:<10}             : {n_benign}")
print(f"  {LABEL_MALIGNANT:<10}             : {n_malignant}")
print(f"  {LABEL_EXCLUDE:<10}             : {n_exclude}")
print(f"Total kandidat final          : {n_candidates}")

# --- bandingkan dengan ekspektasi, JANGAN dipaksakan ----------------------
if n_candidates != EXPECTED_CANDIDATES or n_benign != EXPECTED_BENIGN \
        or n_malignant != EXPECTED_MALIGNANT:
    print("\n" + "!" * 70)
    print("PERINGATAN: JUMLAH KANDIDAT TIDAK SESUAI EKSPETASI EKSTRAKSI")
    print("!" * 70)
    print(f"  {'metrik':<12} {'harapan':>10} {'aktual':>10}")
    print(f"  {'Benign':<12} {EXPECTED_BENIGN:>10} {n_benign:>10}")
    print(f"  {'Malignant':<12} {EXPECTED_MALIGNANT:>10} {n_malignant:>10}")
    print(f"  {'Total':<12} {EXPECTED_CANDIDATES:>10} {n_candidates:>10}")
    print("\nAngka AKTUAL yang dipakai, angka harapan tidak dipaksakan.")
    print("Periksa scripts/03_extract_pylidc_nodules.py bila ini tidak disengaja.")
else:
    print("\nSesuai ekspektasi ekstraksi sebelumnya "
          f"({EXPECTED_BENIGN} / {EXPECTED_MALIGNANT} / {EXPECTED_CANDIDATES}).")

# buang Exclude (satu-satunya penghapusan di notebook ini)
df = df[df["LabelBinary"].notna()].copy()
df["LabelBinary"] = df["LabelBinary"].astype(np.int8)

print(f"\nSetelah filter: {len(df)} baris, "
      f"{df['PatientID'].nunique()} PatientID, "
      f"{df['LabelBinary'].value_counts().to_dict()}")


Baris tidak cocok dengan aturan label : 0 dari 837
Kolom Label konsisten dengan aturan threshold.

Filter label
----------------------------------------------------------------------
Total kandidat (semua baris) : 837
  Benign                 : 381
  Malignant              : 143
  Exclude                : 313
Total kandidat final          : 524

Sesuai ekspektasi ekstraksi sebelumnya (381 / 143 / 524).

Setelah filter: 524 baris, 225 PatientID, {0: 381, 1: 143}


## 5. Duplicate Check

Duplikasi di sini berarti baris annotation yang sama tercatat lebih dari
sekali. Duplikasi tidak dihapus otomatis; jika ditemukan, proses dihentikan
karena dapat merusak hitungan fold dan klassifikasi.

In [8]:
# --- CELL 5 - CEK DUPLIKASI ---------------------------------------------
duplicate_problems = []

print("CEK DUPLIKASI")
print("-" * 70)

# (a) NoduleClusterID
dup_nc = df[df["NoduleClusterID"].duplicated(keep=False)] \
    .sort_values("NoduleClusterID")
print(f"NoduleClusterID duplikat : {int(df['NoduleClusterID'].duplicated().sum())} baris")
if len(dup_nc):
    display(dup_nc[["PatientID", "SeriesInstanceUID", "NoduleClusterID",
                    "MeanMalignancy", "Label"]])
    duplicate_problems.append(
        f"NoduleClusterID duplikat pada {len(dup_nc)} baris")

# (b) kombinasi PatientID + SeriesInstanceUID + NoduleClusterID
key = ["PatientID", "SeriesInstanceUID", "NoduleClusterID"]
dup_combo = df[df.duplicated(key, keep=False)].sort_values(key)
print(f"\nPatientID+SeriesInstanceUID+NoduleClusterID duplikat : "
      f"{int(df.duplicated(key).sum())} baris")
if len(dup_combo):
    display(dup_combo[[*key, "MeanMalignancy", "Label"]])
    duplicate_problems.append(
        f"kunci gabungan duplikat pada {len(dup_combo)} baris")

# (c) sanity: NoduleID asli (jika ada) ikut dipastikan unik
if "NoduleID" in df.columns:
    dup_nid = int(df["NoduleID"].duplicated().sum())
    print(f"\nNoduleID duplikat : {dup_nid}")
    if dup_nid:
        duplicate_problems.append(f"NoduleID duplikat pada {dup_nid} baris")

print(f"\nJumlah baris unik pada kunci gabungan : {len(df.drop_duplicates(key))} "
      f"dari {len(df)}")

if duplicate_problems:
    print("\n" + "=" * 70)
    print("DUPLIKASI DITEMUKAN - SPLIT DIBATALKAN")
    print("=" * 70)
    for p in duplicate_problems:
        print(f"  - {p}")
    print("\nData TIDAK dihapus otomatis. Tentukan dulu perilaku yang diinginkan,")
    print("lalu jalankan ulang notebook.")
    raise ValueError(
        "Duplikasi annotation ditemukan. Proses dihentikan sebelum split."
    )
print("\nTidak ada duplikasi. Lanjut ke stratified group split.")


CEK DUPLIKASI
----------------------------------------------------------------------
NoduleClusterID duplikat : 0 baris

PatientID+SeriesInstanceUID+NoduleClusterID duplikat : 0 baris

NoduleID duplikat : 0

Jumlah baris unik pada kunci gabungan : 524 dari 524

Tidak ada duplikasi. Lanjut ke stratified group split.


## 6. Stratified Group 5-Fold Split

Pembagian memakai `StratifiedGroupKFold(n_splits=5, shuffle=True,
random_state=42)` dengan:

- `X` = kandidat (baris annotation)
- `y` = `LabelBinary`
- `groups` = **`PatientID`**

**Tujuan anti-leakage:** satu pasien hanya boleh muncul pada **satu** fold, dan
karena itu tidak mungkin muncul di train dan validation pada fold yang sama.

`random_train_test_split` **tidak** dipakai karena bisa memisahkan sampel
pasien yang sama ke train dan validation.

In [9]:
# --- CELL 6 - STRATIFIED GROUP KFOLD -------------------------------------
try:
    from sklearn.model_selection import StratifiedGroupKFold
    SKLEARN_AVAILABLE = True
except ImportError as exc:
    SKLEARN_AVAILABLE = False
    raise ImportError(
        "scikit-learn tidak tersedia, sehingga StratifiedGroupKFold tidak "
        "bisa dipakai.\n\n"
        "Pasang sesuai requirements.txt:\n"
        "    .venv\\Scripts\\python -m pip install -r requirements.txt\n\n"
        "requirements.txt memuat scikit-learn==1.6.1.\n"
        "JANGAN mengganti dengan train_test_split: itu berisiko leakage "
        "pada level pasien.\n\n"
        f"Detail: {exc}"
    )

import sklearn
print(f"scikit-learn : {sklearn.__version__}")
print(f"N_SPLITS     : {N_SPLITS}")
print(f"RANDOM_STATE : {RANDOM_STATE}")
print(f"SHUFFLE      : {SHUFFLE}")
print(f"groups       : PatientID")

X = df.drop(columns=["LabelBinary"])       # X tidak dipakai sklearn, tapi eksplisit
y = df["LabelBinary"]
groups = df["PatientID"]

assert len(df) == len(y) == len(groups)
assert df["LabelBinary"].isin([0, 1]).all(), "LabelBinary harus 0 atau 1"

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=SHUFFLE,
                            random_state=RANDOM_STATE)

fold_col = np.zeros(len(df), dtype=np.int64)
for fold, (_, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    fold_col[val_idx] = fold

df["Fold"] = fold_col
df["GroupID"] = df["PatientID"]           # alias eksplisit untuk audit

print("\nFOLD TERBUAT")
print("-" * 70)
for f in range(1, N_SPLITS + 1):
    n_fold = int((df["Fold"] == f).sum())
    n_pat = df.loc[df["Fold"] == f, "PatientID"].nunique()
    print(f"  Fold {f}: {n_fold:>4} sample, {n_pat:>3} PatientID")

assert set(df["Fold"].unique()) <= set(range(1, N_SPLITS + 1)), \
    f"Fold di luar 1..{N_SPLITS}"
print(f"\nTotal sample : {len(df)}")
print(f"Nilai Fold   : {[int(v) for v in sorted(df['Fold'].unique())]}")


scikit-learn : 1.6.1
N_SPLITS     : 5
RANDOM_STATE : 42
SHUFFLE      : True
groups       : PatientID

FOLD TERBUAT
----------------------------------------------------------------------
  Fold 1:   82 sample,  40 PatientID
  Fold 2:  126 sample,  47 PatientID
  Fold 3:  106 sample,  46 PatientID
  Fold 4:   97 sample,  45 PatientID
  Fold 5:  113 sample,  47 PatientID

Total sample : 524
Nilai Fold   : [1, 2, 3, 4, 5]


## 7. Leakage Audit

Audit otomatis untuk memastikan tidak ada kebocoran data antar fold.

Diperiksa:

- `intersection(train_patients, val_patients)` harus kosong
- setiap `PatientID` hanya boleh memiliki satu `Fold`
- semua sample harus memiliki `Fold`
- tidak ada sample yang masuk lebih dari satu fold

In [10]:
# --- CELL 7 - AUDIT LEAKAGE ---------------------------------------------
leakage_problems = []
rows = []

print("AUDIT LEAKAGE")
print("=" * 104)
for f in range(1, N_SPLITS + 1):
    train = df[df["Fold"] != f]
    val = df[df["Fold"] == f]

    train_pat = set(train["PatientID"])
    val_pat = set(val["PatientID"])
    overlap = train_pat & val_pat

    rows.append({
        "Fold": f,
        "Train Samples": len(train),
        "Val Samples": len(val),
        "Train Benign": int((train["LabelBinary"] == 0).sum()),
        "Train Malignant": int((train["LabelBinary"] == 1).sum()),
        "Val Benign": int((val["LabelBinary"] == 0).sum()),
        "Val Malignant": int((val["LabelBinary"] == 1).sum()),
        "Train Patients": len(train_pat),
        "Val Patients": len(val_pat),
        "Patient Overlap": len(overlap),
    })
    flag = "OK" if not overlap else "LEAKAGE"
    print(f"Fold {f}: train {len(train):>4} sample / {len(train_pat):>3} pasien | "
          f"val {len(val):>4} sample / {len(val_pat):>3} pasien | "
          f"overlap {len(overlap)}  [{flag}]")
    if overlap:
        leakage_problems.append(
            f"Fold {f}: {len(overlap)} PatientID muncul di train dan validation")

leakage_table = pd.DataFrame(rows)
display(leakage_table)

# --- pemeriksaan tambahan -------------------------------------------------
print("\nPEMERIKSAAN TAMBAHAN")
print("-" * 70)

# setiap PatientID satu fold
fold_per_patient = df.groupby("PatientID")["Fold"].nunique()
multi_fold_patient = int((fold_per_patient > 1).sum())
print(f"PatientID dengan >1 Fold        : {multi_fold_patient}")
if multi_fold_patient:
    leakage_problems.append(
        f"{multi_fold_patient} PatientID tersebar pada lebih dari satu fold")

# semua sample punya fold
n_no_fold = int(df["Fold"].isna().sum())
print(f"Sample tanpa Fold               : {n_no_fold}")
if n_no_fold:
    leakage_problems.append(f"{n_no_fold} sample tanpa nilai Fold")

# satu sample satu fold (dicek lewat index, bukan kolom)
dup_index = df.index[df.index.duplicated()].tolist()
print(f"Baris dengan index duplikat     : {len(dup_index)}")
if dup_index:
    leakage_problems.append(f"{len(dup_index)} baris terindeks dua kali")

# tiap fold harus ada train dan validation
for f in range(1, N_SPLITS + 1):
    if not (df["Fold"] == f).any():
        leakage_problems.append(f"Fold {f} tidak berisi sample")

print("\n" + "=" * 70)
if leakage_problems:
    print("LEAKAGE TERDETEKSI")
    print("=" * 70)
    for p in leakage_problems:
        print(f"  - {p}")
    raise ValueError("Leakage antar fold. Split tidak valid.")
print("HASIL: tidak ada leakage. Train dan validation terpisah pada level pasien.")


AUDIT LEAKAGE
Fold 1: train  442 sample / 185 pasien | val   82 sample /  40 pasien | overlap 0  [OK]
Fold 2: train  398 sample / 178 pasien | val  126 sample /  47 pasien | overlap 0  [OK]
Fold 3: train  418 sample / 179 pasien | val  106 sample /  46 pasien | overlap 0  [OK]
Fold 4: train  427 sample / 180 pasien | val   97 sample /  45 pasien | overlap 0  [OK]
Fold 5: train  411 sample / 178 pasien | val  113 sample /  47 pasien | overlap 0  [OK]


,Fold,Train Samples,Val Samples,Train Benign,Train Malignant,Val Benign,Val Malignant,Train Patients,Val Patients,Patient Overlap
0,1,442,82,324,118,57,25,185,40,0
1,2,398,126,287,111,94,32,178,47,0
2,3,418,106,307,111,74,32,179,46,0
3,4,427,97,312,115,69,28,180,45,0
4,5,411,113,294,117,87,26,178,47,0



PEMERIKSAAN TAMBAHAN
----------------------------------------------------------------------
PatientID dengan >1 Fold        : 0
Sample tanpa Fold               : 0
Baris dengan index duplikat     : 0

HASIL: tidak ada leakage. Train dan validation terpisah pada level pasien.


## 8. Fold Summary

Tabel ringkasan per fold beserta proporsi kelas.

**Tidak** ada oversampling atau undersampling di notebook ini. Ketidakseimbangan
kelas ditangani pada tahap training, bukan di sini.

In [11]:
# --- CELL 8 - RINGKASAN FOLD --------------------------------------------
summary = []
for f in range(1, N_SPLITS + 1):
    train = df[df["Fold"] != f]
    val = df[df["Fold"] == f]
    tr_b = int((train["LabelBinary"] == 0).sum())
    tr_m = int((train["LabelBinary"] == 1).sum())
    va_b = int((val["LabelBinary"] == 0).sum())
    va_m = int((val["LabelBinary"] == 1).sum())
    summary.append({
        "Fold": f,
        "Train Samples": len(train),
        "Val Samples": len(val),
        "Train Benign": tr_b,
        "Train Malignant": tr_m,
        "Val Benign": va_b,
        "Val Malignant": va_m,
        "Train Patients": train["PatientID"].nunique(),
        "Val Patients": val["PatientID"].nunique(),
        "Train %Malignant": round(tr_m / len(train) * 100, 2) if len(train) else np.nan,
        "Val %Malignant": round(va_m / len(val) * 100, 2) if len(val) else np.nan,
        "Val %Benign": round(va_b / len(val) * 100, 2) if len(val) else np.nan,
    })

fold_summary = pd.DataFrame(summary)

print("RINGKASAN FOLD")
print("=" * 116)
display(fold_summary)
print(f"\nProporsi kelas Benign/Malignant:")
print(f"  keseluruhan : Benign {(df.LabelBinary == 0).mean() * 100:.2f}%  "
      f"Malignant {(df.LabelBinary == 1).mean() * 100:.2f}%")
print(f"  rentang val malignant : "
      f"{fold_summary['Val %Malignant'].min():.2f}% .. "
      f"{fold_summary['Val %Malignant'].max():.2f}%")
print("\nTidak ada oversampling/undersampling pada notebook ini.")


RINGKASAN FOLD


,Fold,Train Samples,Val Samples,Train Benign,Train Malignant,Val Benign,Val Malignant,Train Patients,Val Patients,Train %Malignant,Val %Malignant,Val %Benign
0,1,442,82,324,118,57,25,185,40,26.70,30.49,69.51
1,2,398,126,287,111,94,32,178,47,27.89,25.40,74.60
2,3,418,106,307,111,74,32,179,46,26.56,30.19,69.81
3,4,427,97,312,115,69,28,180,45,26.93,28.87,71.13
4,5,411,113,294,117,87,26,178,47,28.47,23.01,76.99



Proporsi kelas Benign/Malignant:
  keseluruhan : Benign 72.71%  Malignant 27.29%
  rentang val malignant : 23.01% .. 30.49%

Tidak ada oversampling/undersampling pada notebook ini.


## 9. Save Dataset Manifest

Manifest final berisi identitas, label, koordinat bbox, dan nomor fold.
Path DICOM **tidak** dimasukkan karena belum diperlukan pada tahap ini.

In [12]:
# --- CELL 9 - MANIFEST + OUTPUT ------------------------------------------
MANIFEST_COLUMNS = [
    "PatientID",
    "SeriesInstanceUID",
    "NoduleClusterID",
    "MeanMalignancy",
    "NReaders",
    "Label",
    "LabelBinary",
    "BBox_X_Min", "BBox_X_Max",
    "BBox_Y_Min", "BBox_Y_Max",
    "BBox_Z_Min", "BBox_Z_Max",
    "BBox_Z_Slices",
    "Fold",
    "GroupID",
]

missing_cols = [c for c in MANIFEST_COLUMNS if c not in df.columns]
if missing_cols:
    raise KeyError(f"Kolom manifest tidak tersedia: {missing_cols}")

manifest = df[MANIFEST_COLUMNS].copy()
manifest = manifest.sort_values(["Fold", "PatientID", "NoduleClusterID"]) \
                    .reset_index(drop=True)
manifest["Fold"] = manifest["Fold"].astype(int)
manifest["LabelBinary"] = manifest["LabelBinary"].astype(int)
manifest["GroupID"] = manifest["GroupID"].astype(str)

fold_assignments = manifest[["PatientID", "GroupID", "SeriesInstanceUID",
                             "NoduleClusterID", "Label", "LabelBinary",
                             "Fold"]].copy()

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = SPLIT_DIR / MANIFEST_FILENAME
fold_assign_path = SPLIT_DIR / FOLD_ASSIGN_FILENAME
fold_summary_path = SPLIT_DIR / FOLD_SUMMARY_FILENAME
split_config_path = SPLIT_DIR / SPLIT_CONFIG_FILENAME

manifest.to_csv(manifest_path, index=False)
fold_assignments.to_csv(fold_assign_path, index=False)
fold_summary.to_csv(fold_summary_path, index=False)

split_config = {
    "source_annotation": str(ANNOTATION_RELPATH),
    "generated_by": "notebooks/02_dataset_preparation.ipynb",
    "strategy": "StratifiedGroupKFold",
    "n_splits": N_SPLITS,
    "shuffle": SHUFFLE,
    "random_state": RANDOM_STATE,
    "grouping_variable": "PatientID",
    "x_column_used": "tidak dipakai sklearn; metadata baris annotation",
    "y_column": "LabelBinary",
    "label_rule": {
        "benign": f"MeanMalignancy <= {LABEL_THRESHOLD_BENIGN_MAX}",
        "exclude": f"{LABEL_THRESHOLD_BENIGN_MAX} < MeanMalignancy < "
                   f"{LABEL_THRESHOLD_MALIGNANT_MIN}",
        "malignant": f"MeanMalignancy >= {LABEL_THRESHOLD_MALIGNANT_MIN}",
        "excluded_from_dataset": LABEL_EXCLUDE,
        "binary_map": {str(k): int(v) for k, v in LABEL_BINARY_MAP.items()},
    },
    "nodule_cluster_id_source": f"NoduleClusterID <- {CLUSTER_ID_SOURCE} "
                                 f"(file sumber tidak punya kolom "
                                 f"NoduleClusterID)",
    "counts": {
        "total_candidates": int(len(manifest)),
        "benign": int((manifest["LabelBinary"] == 0).sum()),
        "malignant": int((manifest["LabelBinary"] == 1).sum()),
        "patients": int(manifest["PatientID"].nunique()),
        "series": int(manifest["SeriesInstanceUID"].nunique()),
        "nodules": int(manifest["NoduleClusterID"].nunique()),
    },
    "oversampling": "tidak dilakukan pada tahap ini",
    "outputs": {
        "manifest": MANIFEST_FILENAME,
        "fold_assignments": FOLD_ASSIGN_FILENAME,
        "fold_summary": FOLD_SUMMARY_FILENAME,
    },
}
with open(split_config_path, "w", encoding="utf-8") as f:
    json.dump(split_config, f, indent=2, ensure_ascii=False, default=str)

print("OUTPUT TERSIMPAN")
print("-" * 70)
for p in (manifest_path, fold_assign_path, fold_summary_path, split_config_path):
    print(f"  {p.relative_to(PROJECT_ROOT)}  ({p.stat().st_size:,} bytes)")
print(f"\nManifest : {len(manifest)} baris x {len(manifest.columns)} kolom")
print(f"Kolom    : {list(manifest.columns)}")
display(manifest.head(5))


OUTPUT TERSIMPAN
----------------------------------------------------------------------
  dataset\05_splits\nodule_manifest.csv  (110,249 bytes)
  dataset\05_splits\fold_assignments.csv  (93,253 bytes)
  dataset\05_splits\fold_summary.csv  (403 bytes)
  dataset\05_splits\split_config.json  (1,120 bytes)

Manifest : 524 baris x 16 kolom
Kolom    : ['PatientID', 'SeriesInstanceUID', 'NoduleClusterID', 'MeanMalignancy', 'NReaders', 'Label', 'LabelBinary', 'BBox_X_Min', 'BBox_X_Max', 'BBox_Y_Min', 'BBox_Y_Max', 'BBox_Z_Min', 'BBox_Z_Max', 'BBox_Z_Slices', 'Fold', 'GroupID']


,PatientID,SeriesInstanceUID,NoduleClusterID,MeanMalignancy,NReaders,Label,LabelBinary,BBox_X_Min,BBox_X_Max,BBox_Y_Min,BBox_Y_Max,BBox_Z_Min,BBox_Z_Max,BBox_Z_Slices,Fold,GroupID
0,LIDC-IDRI-0001,1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636...,1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636...,4.75,4,Malignant,1,340,391,297,340,86,94,9,1,LIDC-IDRI-0001
1,LIDC-IDRI-0007,1.3.6.1.4.1.14519.5.2.1.6279.6001.272348349298...,1.3.6.1.4.1.14519.5.2.1.6279.6001.272348349298...,4.75,4,Malignant,1,263,315,173,217,101,113,13,1,LIDC-IDRI-0007
2,LIDC-IDRI-0007,1.3.6.1.4.1.14519.5.2.1.6279.6001.272348349298...,1.3.6.1.4.1.14519.5.2.1.6279.6001.272348349298...,5.00,1,Malignant,1,255,280,279,305,118,129,12,1,LIDC-IDRI-0007
3,LIDC-IDRI-0049,1.3.6.1.4.1.14519.5.2.1.6279.6001.239358021703...,1.3.6.1.4.1.14519.5.2.1.6279.6001.239358021703...,2.50,2,Benign,0,339,356,116,130,35,38,4,1,LIDC-IDRI-0049
4,LIDC-IDRI-0049,1.3.6.1.4.1.14519.5.2.1.6279.6001.239358021703...,1.3.6.1.4.1.14519.5.2.1.6279.6001.239358021703...,1.50,2,Benign,0,136,159,350,363,51,55,5,1,LIDC-IDRI-0049


## 10. Final Validation

Ringkasan akhir plus assertion untuk kondisi kritis split.

In [13]:
# --- CELL 10 - VALIDASI AKHIR --------------------------------------------
print("VALIDASI AKHIR")
print("=" * 70)
print(f"Total kandidat final   : {len(manifest)}")
print(f"  Benign               : {int((manifest['LabelBinary'] == 0).sum())}")
print(f"  Malignant            : {int((manifest['LabelBinary'] == 1).sum())}")
print(f"PatientID unik         : {manifest['PatientID'].nunique()}")
print(f"SeriesInstanceUID unik : {manifest['SeriesInstanceUID'].nunique()}")
print(f"NoduleClusterID unik   : {manifest['NoduleClusterID'].nunique()}")

print("\nSample per fold:")
for f in range(1, N_SPLITS + 1):
    print(f"  Fold {f}: {int((manifest['Fold'] == f).sum()):>4} sample, "
          f"{manifest.loc[manifest['Fold'] == f, 'PatientID'].nunique():>3} PatientID")
print(f"  Total  : {len(manifest)} sample, {manifest['PatientID'].nunique()} PatientID")

print("\nLeakage check:")
overlap_total = int(leakage_table["Patient Overlap"].sum())
print(f"  total PatientID overlap train/validation : {overlap_total}")
print(f"  PatientID dengan >1 Fold                 : {multi_fold_patient}")
print(f"  sample tanpa Fold                        : {n_no_fold}")
print("  hasil                                    : "
      + ("BERSIH" if overlap_total == 0 and multi_fold_patient == 0
         and n_no_fold == 0 else "BERMASALAH"))

print("\nOutput file:")
for p in (manifest_path, fold_assign_path, fold_summary_path, split_config_path):
    print(f"  {p.relative_to(PROJECT_ROOT)}")

# --- assertion kondisi kritis --------------------------------------------
print("\nASSERTION")
print("-" * 70)
assertions = []


def assert_that(name, condition):
    ok = bool(condition)
    assertions.append((name, ok))
    print(f"  [{'OK ' if ok else 'GAGAL'}] {name}")
    if not ok:
        raise AssertionError(f"Assertion gagal: {name}")


assert_that(f"semua Fold berada di 1..{N_SPLITS}",
            manifest["Fold"].isin(range(1, N_SPLITS + 1)).all())
assert_that("tidak ada PatientID dengan lebih dari satu Fold",
            manifest.groupby("PatientID")["Fold"].nunique().max() == 1)
assert_that("tidak ada sample tanpa Fold", manifest["Fold"].notna().all())
assert_that("tidak ada overlap PatientID train/validation",
            overlap_total == 0)
assert_that("LabelBinary hanya 0 atau 1",
            set(manifest["LabelBinary"].unique()) <= {0, 1})
assert_that(f"jumlah fold tepat {N_SPLITS}",
            manifest["Fold"].nunique() == N_SPLITS)
assert_that("tidak ada Exclude pada manifest",
            (manifest["Label"] != LABEL_EXCLUDE).all())
assert_that("GroupID = PatientID",
            (manifest["GroupID"].astype(str)
             == manifest["PatientID"].astype(str)).all())
assert_that("jumlah baris manifest = total kandidat",
            len(manifest) == n_candidates)

print("\nSeluruh assertion terpenuhi.")
print("\nRINGKASAN")
print("=" * 70)
print(f"  Kandidat   : {len(manifest)} "
      f"(Benign {int((manifest['LabelBinary'] == 0).sum())} / "
      f"Malignant {int((manifest['LabelBinary'] == 1).sum())})")
print(f"  Pasien     : {manifest['PatientID'].nunique()}")
print(f"  Fold       : {N_SPLITS}-fold StratifiedGroupKFold pada PatientID")
print(f"  Leakage    : tidak ada")
print(f"  Resampling : tidak ada")
print(f"  Lokasi     : {SPLIT_RELDIR}")
print("\nLangkah berikutnya: training DenseNet121 per fold, lalu Grad-CAM.")
print("Notebook itu TIDAK ada di file ini.")


VALIDASI AKHIR
Total kandidat final   : 524
  Benign               : 381
  Malignant            : 143
PatientID unik         : 225
SeriesInstanceUID unik : 225
NoduleClusterID unik   : 524

Sample per fold:
  Fold 1:   82 sample,  40 PatientID
  Fold 2:  126 sample,  47 PatientID
  Fold 3:  106 sample,  46 PatientID
  Fold 4:   97 sample,  45 PatientID
  Fold 5:  113 sample,  47 PatientID
  Total  : 524 sample, 225 PatientID

Leakage check:
  total PatientID overlap train/validation : 0
  PatientID dengan >1 Fold                 : 0
  sample tanpa Fold                        : 0
  hasil                                    : BERSIH

Output file:
  dataset\05_splits\nodule_manifest.csv
  dataset\05_splits\fold_assignments.csv
  dataset\05_splits\fold_summary.csv
  dataset\05_splits\split_config.json

ASSERTION
----------------------------------------------------------------------
  [OK ] semua Fold berada di 1..5
  [OK ] tidak ada PatientID dengan lebih dari satu Fold
  [OK ] tidak ada sa